<p align="center">
  <img src="logo.png" width="250" alt="BioGuard Logo">
</p>

# 🌎 BioGuard — FIAP Global Solution 2026
## Monitoramento Inteligente de Queimadas e Impacto na Fauna Brasileira

Este notebook utiliza:
- 🔥 Dados do INPE BDQueimadas
- 🐾 Base local da IUCN Red List
- 🗺️ Mapas interativos
- 📊 Dashboards analíticos
- 🧠 Índice de vulnerabilidade ambiental


# 📦 IMPORTS

In [12]:
import pandas as pd
import numpy as np
import folium
import plotly.express as px
import matplotlib.pyplot as plt
import tabulate


from folium.plugins import (
    HeatMap,
    MarkerCluster,
    Fullscreen,
    MiniMap
)

from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)



# 📂 Carregamento dos dados do INPE


In [13]:
arquivo = "bdqueimadas.csv"

df = pd.read_csv(arquivo)

# Padronizar colunas
df.columns = df.columns.str.lower().str.strip()

print("📋 Colunas disponíveis:\n")
print(df.columns.tolist())

df.head()


📋 Colunas disponíveis:

['datahora', 'satelite', 'pais', 'estado', 'municipio', 'bioma', 'diasemchuva', 'precipitacao', 'riscofogo', 'frp', 'latitude', 'longitude']


,datahora,satelite,pais,estado,municipio,bioma,diasemchuva,precipitacao,riscofogo,frp,latitude,longitude
0,2026/01/01 00:00:00,GOES-19,Brasil,PIAUÍ,REDENÇÃO DO GURGUÉIA,Cerrado,2.0,0.66,0.06,71.9,-9.563100,-44.603000
1,2026/01/01 00:00:00,GOES-19,Brasil,PIAUÍ,REDENÇÃO DO GURGUÉIA,Cerrado,2.0,1.39,0.07,77.3,-9.582100,-44.600700
2,2026/01/01 00:05:00,METOP-B,Brasil,PARÁ,MEDICILÂNDIA,Amazônia,3.0,0.00,0.01,NaN,-3.032000,-53.102501
3,2026/01/01 00:05:00,METOP-B,Brasil,ESPÍRITO SANTO,SÃO MATEUS,Mata Atlântica,-999.0,0.00,0.12,NaN,-18.607599,-39.992199
4,2026/01/01 00:05:00,METOP-B,Brasil,PARÁ,RONDON DO PARÁ,Amazônia,4.0,0.00,0.36,NaN,-4.630400,-47.932800



# 🧹 Limpeza e preparação dos dados


In [14]:
rename_map = {}

if "latitude" in df.columns:
    rename_map["latitude"] = "lat"

if "longitude" in df.columns:
    rename_map["longitude"] = "lon"

df = df.rename(columns=rename_map)

# Detectar coluna de bioma
coluna_bioma = None

for col in df.columns:
    if "bioma" in col:
        coluna_bioma = col
        break

print(f"🌱 Coluna de bioma encontrada: {coluna_bioma}")


print(f"🔥 Registros: {len(df)}")

# Remover nulos
df = df.dropna(subset=["lat", "lon"])

df.head()


🌱 Coluna de bioma encontrada: bioma
🔥 Registros: 277981


,datahora,satelite,pais,estado,municipio,bioma,diasemchuva,precipitacao,riscofogo,frp,lat,lon
0,2026/01/01 00:00:00,GOES-19,Brasil,PIAUÍ,REDENÇÃO DO GURGUÉIA,Cerrado,2.0,0.66,0.06,71.9,-9.563100,-44.603000
1,2026/01/01 00:00:00,GOES-19,Brasil,PIAUÍ,REDENÇÃO DO GURGUÉIA,Cerrado,2.0,1.39,0.07,77.3,-9.582100,-44.600700
2,2026/01/01 00:05:00,METOP-B,Brasil,PARÁ,MEDICILÂNDIA,Amazônia,3.0,0.00,0.01,NaN,-3.032000,-53.102501
3,2026/01/01 00:05:00,METOP-B,Brasil,ESPÍRITO SANTO,SÃO MATEUS,Mata Atlântica,-999.0,0.00,0.12,NaN,-18.607599,-39.992199
4,2026/01/01 00:05:00,METOP-B,Brasil,PARÁ,RONDON DO PARÁ,Amazônia,4.0,0.00,0.36,NaN,-4.630400,-47.932800



# 🐾 Base local de espécies ameaçadas


In [15]:
dados_especies = [
    # 🌿 PANTANAL
    {"especie": "Panthera onca", "nome_comum": "Onça-pintada", "bioma": "Pantanal", "categoria_iucn": "NT"},
    {"especie": "Anodorhynchus hyacinthinus", "nome_comum": "Arara-azul", "bioma": "Pantanal", "categoria_iucn": "VU"},
    {"especie": "Myrmecophaga tridactyla", "nome_comum": "Tamanduá-bandeira", "bioma": "Pantanal", "categoria_iucn": "VU"},
    {"especie": "Chrysocyon brachyurus", "nome_comum": "Lobo-guará", "bioma": "Pantanal", "categoria_iucn": "NT"},
    {"especie": "Caiman yacare", "nome_comum": "Jacaré-do-pantanal", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Jabiru mycteria", "nome_comum": "Tuiuiú", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Hydrochoerus hydrochaeris", "nome_comum": "Capivara", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Tapirus terrestris", "nome_comum": "Anta", "bioma": "Pantanal", "categoria_iucn": "VU"},
    {"especie": "Pteronura brasiliensis", "nome_comum": "Ariranha", "bioma": "Pantanal", "categoria_iucn": "EN"},
    {"especie": "Leopardus pardalis", "nome_comum": "Jaguatirica", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Dasyprocta leporina", "nome_comum": "Cutia", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Tayassu pecari", "nome_comum": "Queixada", "bioma": "Pantanal", "categoria_iucn": "NT"},
    {"especie": "Rhea americana", "nome_comum": "Ema", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Phrynops hilarii", "nome_comum": "Cágado-de-barbiga", "bioma": "Pantanal", "categoria_iucn": "LC"},
    {"especie": "Caiman crocodilus", "nome_comum": "Jacaré-açu", "bioma": "Pantanal", "categoria_iucn": "LC"},

    # 🌳 AMAZÔNIA
    {"especie": "Saguinus bicolor", "nome_comum": "Sauim-de-coleira", "bioma": "Amazônia", "categoria_iucn": "CR"},
    {"especie": "Ateles paniscus", "nome_comum": "Macaco-aranha", "bioma": "Amazônia", "categoria_iucn": "VU"},
    {"especie": "Cacajao calvus", "nome_comum": "Uacari-branco", "bioma": "Amazônia", "categoria_iucn": "VU"},
    {"especie": "Inia geoffrensis", "nome_comum": "Boto-cor-de-rosa", "bioma": "Amazônia", "categoria_iucn": "EN"},
    {"especie": "Pteronura brasiliensis", "nome_comum": "Ariranha", "bioma": "Amazônia", "categoria_iucn": "EN"},
    {"especie": "Harpia harpyja", "nome_comum": "Gavião-real", "bioma": "Amazônia", "categoria_iucn": "NT"},
    {"especie": "Podocnemis expansa", "nome_comum": "Tartaruga-da-amazônia", "bioma": "Amazônia", "categoria_iucn": "VU"},
    {"especie": "Alouatta seniculus", "nome_comum": "Guariba-vermelho", "bioma": "Amazônia", "categoria_iucn": "LC"},
    {"especie": "Cebus albifrons", "nome_comum": "Macaco-prego", "bioma": "Amazônia", "categoria_iucn": "LC"},
    {"especie": "Eira barbara", "nome_comum": "Irara", "bioma": "Amazônia", "categoria_iucn": "LC"},
    {"especie": "Phyllostomus hastatus", "nome_comum": "Morcego-lanceolado", "bioma": "Amazônia", "categoria_iucn": "LC"},
    {"especie": "Arapaima gigas", "nome_comum": "Pirarucu", "bioma": "Amazônia", "categoria_iucn": "EN"},
    {"especie": "Dendrobates tinctorius", "nome_comum": "Rã-venenosa", "bioma": "Amazônia", "categoria_iucn": "LC"},
    {"especie": "Myrmecophaga tridactyla", "nome_comum": "Tamanduá-bandeira", "bioma": "Amazônia", "categoria_iucn": "VU"},
    {"especie": "Panthera onca", "nome_comum": "Onça-pintada", "bioma": "Amazônia", "categoria_iucn": "NT"},

    # 🌾 CERRADO
    {"especie": "Chrysocyon brachyurus", "nome_comum": "Lobo-guará", "bioma": "Cerrado", "categoria_iucn": "NT"},
    {"especie": "Pseudalopex vetulus", "nome_comum": "Raposa-do-campo", "bioma": "Cerrado", "categoria_iucn": "NT"},
    {"especie": "Ozotoceros bezoarticus", "nome_comum": "Veado-campeiro", "bioma": "Cerrado", "categoria_iucn": "VU"},
    {"especie": "Leopardus tigrinus", "nome_comum": "Gato-do-mato-pequeno", "bioma": "Cerrado", "categoria_iucn": "VU"},
    {"especie": "Rhea americana", "nome_comum": "Ema", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Cyanocorax cristatellus", "nome_comum": "Gralha-do-cerrado", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Caiman latirostris", "nome_comum": "Jacaré-do-papo-amarelo", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Tapirus terrestris", "nome_comum": "Anta", "bioma": "Cerrado", "categoria_iucn": "VU"},
    {"especie": "Hydrochoerus hydrochaeris", "nome_comum": "Capivara", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Tayassu pecari", "nome_comum": "Queixada", "bioma": "Cerrado", "categoria_iucn": "NT"},
    {"especie": "Dasyprocta leporina", "nome_comum": "Cutia", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Crotalus durissus", "nome_comum": "Cascavel", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Eira barbara", "nome_comum": "Irara", "bioma": "Cerrado", "categoria_iucn": "LC"},
    {"especie": "Alouatta caraya", "nome_comum": "Bugio-preto", "bioma": "Cerrado", "categoria_iucn": "NT"},
    {"especie": "Puma concolor", "nome_comum": "Onça-parda", "bioma": "Cerrado", "categoria_iucn": "LC"},

    # 🌴 MATA ATLÂNTICA
    {"especie": "Leontopithecus rosalia", "nome_comum": "Mico-leão-dourado", "bioma": "Mata Atlântica", "categoria_iucn": "EN"},
    {"especie": "Brachyteles arachnoides", "nome_comum": "Muriqui-do-sul", "bioma": "Mata Atlântica", "categoria_iucn": "CR"},
    {"especie": "Procnias nudicollis", "nome_comum": "Araponga", "bioma": "Mata Atlântica", "categoria_iucn": "VU"},
    {"especie": "Leopardus tigrinus", "nome_comum": "Gato-do-mato", "bioma": "Mata Atlântica", "categoria_iucn": "VU"},
    {"especie": "Ramphastos toco", "nome_comum": "Tucanuçu", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Didelphis aurita", "nome_comum": "Gambá-de-orelha-preta", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Callithrix penicillata", "nome_comum": "Sagui", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Tamandua tetradactyla", "nome_comum": "Tamanduá-mirim", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Trichechus manatus", "nome_comum": "Peixe-boi-marinho", "bioma": "Mata Atlântica", "categoria_iucn": "VU"},
    {"especie": "Buteogallus coronatus", "nome_comum": "Gavião-pombo", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Cyclarhis gujanensis", "nome_comum": "Pitiguari", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Puma concolor", "nome_comum": "Onça-parda", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Dasyprocta leporina", "nome_comum": "Cutia", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Phyllomedusa bicolor", "nome_comum": "Perereca-gigante", "bioma": "Mata Atlântica", "categoria_iucn": "LC"},
    {"especie": "Alouatta guariba", "nome_comum": "Bugio-ruivo", "bioma": "Mata Atlântica", "categoria_iucn": "CR"},

    # 🌵 CAATINGA
    {"especie": "Callithrix jacchus", "nome_comum": "Sagui-do-nordeste", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Pseudalopex vetulus", "nome_comum": "Raposa-do-campo", "bioma": "Caatinga", "categoria_iucn": "NT"},
    {"especie": "Leopardus tigrinus", "nome_comum": "Gato-do-mato", "bioma": "Caatinga", "categoria_iucn": "VU"},
    {"especie": "Dendropsophus microcephalus", "nome_comum": "Perereca", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Tyrannus melancholicus", "nome_comum": "Suiriri", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Crotalus durissus", "nome_comum": "Cascavel", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Tapirus terrestris", "nome_comum": "Anta", "bioma": "Caatinga", "categoria_iucn": "VU"},
    {"especie": "Hydrochoerus hydrochaeris", "nome_comum": "Capivara", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Caiman latirostris", "nome_comum": "Jacaré", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Dasyprocta leporina", "nome_comum": "Cutia", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Rhea americana", "nome_comum": "Ema", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Cynomops planirostris", "nome_comum": "Morcego-de-focinho-curto", "bioma": "Caatinga", "categoria_iucn": "LC"},
    {"especie": "Tayassu pecari", "nome_comum": "Queixada", "bioma": "Caatinga", "categoria_iucn": "NT"},
    {"especie": "Ozotoceros bezoarticus", "nome_comum": "Veado-campeiro", "bioma": "Caatinga", "categoria_iucn": "VU"},
    {"especie": "Pygocentrus nattereri", "nome_comum": "Piranha-vermelha", "bioma": "Caatinga", "categoria_iucn": "LC"},

    # 🌾 PAMPA
    {"especie": "Ozotoceros bezoarticus", "nome_comum": "Veado-campeiro", "bioma": "Pampa", "categoria_iucn": "VU"},
    {"especie": "Rhea pennata", "nome_comum": "Ema-do-pampa", "bioma": "Pampa", "categoria_iucn": "NT"},
    {"especie": "Lagostomus maximus", "nome_comum": "Lebre-da-patagônia", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Puma concolor", "nome_comum": "Puma", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Leopardus pardalis", "nome_comum": "Jaguatirica", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Hydrochoerus hydrochaeris", "nome_comum": "Capivara", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Dasyprocta leporina", "nome_comum": "Cutia", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Chrysocyon brachyurus", "nome_comum": "Lobo-guará", "bioma": "Pampa", "categoria_iucn": "NT"},
    {"especie": "Tapirus terrestris", "nome_comum": "Anta", "bioma": "Pampa", "categoria_iucn": "VU"},
    {"especie": "Caiman latirostris", "nome_comum": "Jacaré", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Phrynops hilarii", "nome_comum": "Cágado", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Cynomops planirostris", "nome_comum": "Morcego", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Crotalus durissus", "nome_comum": "Cascavel", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Rhea americana", "nome_comum": "Ema", "bioma": "Pampa", "categoria_iucn": "LC"},
    {"especie": "Tayassu pecari", "nome_comum": "Queixada", "bioma": "Pampa", "categoria_iucn": "NT"}
]

df_especies = pd.DataFrame(dados_especies)

df_especies


,especie,nome_comum,bioma,categoria_iucn
0,Panthera onca,Onça-pintada,Pantanal,NT
1,Anodorhynchus hyacinthinus,Arara-azul,Pantanal,VU
2,Myrmecophaga tridactyla,Tamanduá-bandeira,Pantanal,VU
3,Chrysocyon brachyurus,Lobo-guará,Pantanal,NT
4,Caiman yacare,Jacaré-do-pantanal,Pantanal,LC
...,...,...,...,...
85,Phrynops hilarii,Cágado,Pampa,LC
86,Cynomops planirostris,Morcego,Pampa,LC
87,Crotalus durissus,Cascavel,Pampa,LC
88,Rhea americana,Ema,Pampa,LC


# ⚖️ Índice de risco ambiental

In [ ]:
PESO_IUCN = {
    "CR": 5,
    "EN": 4,
    "VU": 3,
    "NT": 2,
    "LC": 1
}

df_especies["peso_risco"] = df_especies["categoria_iucn"].map(PESO_IUCN)

df_especies

,especie,nome_comum,bioma,categoria_iucn,peso_risco
0,Panthera onca,Onça-pintada,Pantanal,NT,2
1,Anodorhynchus hyacinthinus,Arara-azul,Pantanal,VU,3
2,Myrmecophaga tridactyla,Tamanduá-bandeira,Pantanal,VU,3
3,Chrysocyon brachyurus,Lobo-guará,Pantanal,NT,2
4,Caiman yacare,Jacaré-do-pantanal,Pantanal,LC,1
...,...,...,...,...,...
85,Phrynops hilarii,Cágado,Pampa,LC,1
86,Cynomops planirostris,Morcego,Pampa,LC,1
87,Crotalus durissus,Cascavel,Pampa,LC,1
88,Rhea americana,Ema,Pampa,LC,1


## 🧠 CÁLCULO DE RISCO

In [ ]:
qtd_queimadas = len(df)

media_risco = df_especies["peso_risco"].mean()

indice_vulnerabilidade = qtd_queimadas * media_risco

print(f"🔥 Queimadas registradas: {qtd_queimadas}")
print(f"🐾 Média de risco das espécies: {round(media_risco,2)}")
print(f"🚨 Índice de vulnerabilidade: {round(indice_vulnerabilidade,2)}")

if indice_vulnerabilidade < 500:
    nivel = "Baixo"

elif indice_vulnerabilidade < 2000:
    nivel = "Moderado"

elif indice_vulnerabilidade < 5000:
    nivel = "Alto"

else:
    nivel = "Crítico"

print(f"⚠️ Classificação ambiental: {nivel}")


🔥 Queimadas registradas: 277981
🐾 Média de risco das espécies: 1.86
🚨 Índice de vulnerabilidade: 515809.19
⚠️ Classificação ambiental: Crítico



# 🗺️ Mapa interativo de queimadas


In [18]:


# ============================================================
# 🔍 DETECTAR COLUNAS
# ============================================================

col_lat = None
col_lon = None

for col in df.columns:

    if "lat" in col.lower():
        col_lat = col

    if "lon" in col.lower() or "long" in col.lower():
        col_lon = col

print("Latitude:", col_lat)
print("Longitude:", col_lon)

# ============================================================
# 🚨 VALIDAÇÃO
# ============================================================

if col_lat is None or col_lon is None:

    print("❌ Coordenadas não encontradas!")

else:

    # ========================================================
    # 🧹 LIMPEZA
    # ========================================================

    mapa_df = df.dropna(subset=[col_lat, col_lon])

    print("🔥 Registros usados:", len(mapa_df))

    # ========================================================
    # 🌎 MAPA BASE
    # ========================================================

    mapa = folium.Map(
        location=[-14, -55],
        zoom_start=4,
        tiles="CartoDB positron"
    )

    # ========================================================
    # 🛰️ CONTROLES EXTRAS
    # ========================================================

    Fullscreen().add_to(mapa)

    MiniMap(toggle_display=True).add_to(mapa)

    # ========================================================
    # 🔥 HEATMAP QUEIMADAS
    # ========================================================

    heat_data = mapa_df[[col_lat, col_lon]].values.tolist()

    HeatMap(
        heat_data,
        radius=18,
        blur=12,
        min_opacity=0.4,
        max_zoom=8
    ).add_to(mapa)

    # ========================================================
    # 🐾 ESPÉCIES
    # ========================================================

    especies = [
    # --- PANTANAL ---
    {
        "nome": "🐾 Onça-pintada",
        "lat": -16.5,
        "lon": -56.5,
        "bioma": "Pantanal",
        "risco": "Alto"
    },
    {
        "nome": "🐦 Arara-azul-grande",
        "lat": -18.2,
        "lon": -57.3,
        "bioma": "Pantanal",
        "risco": "Moderado"
    },
    {
        "nome": "🐊 Jacaré-do-papo-amarelo",
        "lat": -19.0,
        "lon": -57.0,
        "bioma": "Pantanal",
        "risco": "Baixo"
    },
    {
        "nome": "🦌 Veado-mateiro",
        "lat": -17.5,
        "lon": -56.0,
        "bioma": "Pantanal",
        "risco": "Alto"
    },
    {
        "nome": "🦦 Lontra",
        "lat": -18.5,
        "lon": -56.8,
        "bioma": "Pantanal",
        "risco": "Crítico"
    },
    {
        "nome": "🐗 Queixada",
        "lat": -16.2,
        "lon": -55.9,
        "bioma": "Pantanal",
        "risco": "Baixo"
    },
    {
        "nome": "🐍 Sucuri",
        "lat": -19.2,
        "lon": -57.1,
        "bioma": "Pantanal",
        "risco": "Alto"
    },
    {
        "nome": "🦜 Arara-canindé",
        "lat": -17.8,
        "lon": -57.5,
        "bioma": "Pantanal",
        "risco": "Moderado"
    },

    # --- AMAZÔNIA ---
    {
        "nome": "🐒 Sauim-de-coleira",
        "lat": -3.1,
        "lon": -60.0,
        "bioma": "Amazônia",
        "risco": "Crítico"
    },
    {
        "nome": "🐒 Macaco-aranha",
        "lat": -5.5,
        "lon": -63.0,
        "bioma": "Amazônia",
        "risco": "Alto"
    },
    {
        "nome": "🦜 Arara-vermelha",
        "lat": -4.0,
        "lon": -62.5,
        "bioma": "Amazônia",
        "risco": "Moderado"
    },
    {
        "nome": "🐢 Tartaruga-da-amazônia",
        "lat": -3.5,
        "lon": -59.8,
        "bioma": "Amazônia",
        "risco": "Alto"
    },
    {
        "nome": "🐆 Ariranha",
        "lat": -8.5,
        "lon": -70.0,
        "bioma": "Amazônia",
        "risco": "Crítico"
    },
    {
        "nome": "🐬 Boto-cor-de-rosa",
        "lat": -4.2,
        "lon": -62.8,
        "bioma": "Amazônia",
        "risco": "Crítico"
    },
    {
        "nome": "🦅 Gavião-real",
        "lat": -5.0,
        "lon": -61.5,
        "bioma": "Amazônia",
        "risco": "Alto"
    },
    {
        "nome": "🐒 Guariba",
        "lat": -6.5,
        "lon": -58.5,
        "bioma": "Amazônia",
        "risco": "Baixo"
    },
    {
        "nome": "🐆 Jaguatirica",
        "lat": -3.8,
        "lon": -64.2,
        "bioma": "Amazônia",
        "risco": "Moderado"
    },

    # --- CERRADO ---
    {
        "nome": "🐺 Lobo-guará",
        "lat": -15.8,
        "lon": -47.9,
        "bioma": "Cerrado",
        "risco": "Alto"
    },
    {
        "nome": "🦔 Tamanduá-bandeira",
        "lat": -16.5,
        "lon": -49.0,
        "bioma": "Cerrado",
        "risco": "Alto"
    },
    {
        "nome": "🐆 Gato-do-mato-pequeno",
        "lat": -14.2,
        "lon": -48.5,
        "bioma": "Cerrado",
        "risco": "Moderado"
    },
    {
        "nome": "🦅 Seriema",
        "lat": -15.0,
        "lon": -47.5,
        "bioma": "Cerrado",
        "risco": "Baixo"
    },
    {
        "nome": "🐗 Cateto",
        "lat": -17.2,
        "lon": -50.1,
        "bioma": "Cerrado",
        "risco": "Baixo"
    },
    {
        "nome": "🦃 Jacutinga",
        "lat": -15.5,
        "lon": -49.5,
        "bioma": "Cerrado",
        "risco": "Crítico"
    },
    {
        "nome": "🐾 Cachorro-do-mato",
        "lat": -16.0,
        "lon": -48.2,
        "bioma": "Cerrado",
        "risco": "Moderado"
    },
    {
        "nome": "🦌 Veado-catingueiro",
        "lat": -14.8,
        "lon": -46.5,
        "bioma": "Cerrado",
        "risco": "Alto"
    },

    # --- MATA ATLÂNTICA ---
    {
        "nome": "🐵 Mico-leão-dourado",
        "lat": -22.5,
        "lon": -42.0,
        "bioma": "Mata Atlântica",
        "risco": "Crítico"
    },
    {
        "nome": "🦜 Muriqui-do-sul",
        "lat": -20.5,
        "lon": -43.5,
        "bioma": "Mata Atlântica",
        "risco": "Crítico"
    },
    {
        "nome": "🐆 Onça-parda",
        "lat": -23.5,
        "lon": -46.6,
        "bioma": "Mata Atlântica",
        "risco": "Alto"
    },
    {
        "nome": "🐸 Sapo-cururu",
        "lat": -22.0,
        "lon": -43.0,
        "bioma": "Mata Atlântica",
        "risco": "Baixo"
    },
    {
        "nome": "🐒 Bugio-ruivo",
        "lat": -21.0,
        "lon": -41.5,
        "bioma": "Mata Atlântica",
        "risco": "Crítico"
    },
    {
        "nome": "🦜 Papagaio-de-peito-roxo",
        "lat": -26.0,
        "lon": -48.5,
        "bioma": "Mata Atlântica",
        "risco": "Crítico"
    },
    {
        "nome": "🦥 Preguiça-de-coleira",
        "lat": -24.5,
        "lon": -47.0,
        "bioma": "Mata Atlântica",
        "risco": "Moderado"
    },
    {
        "nome": "🐸 Perereca-de-folhagem",
        "lat": -23.0,
        "lon": -44.0,
        "bioma": "Mata Atlântica",
        "risco": "Baixo"
    },

    # --- CAATINGA ---
    {
        "nome": "🐕 Raposa-do-campo",
        "lat": -9.5,
        "lon": -40.5,
        "bioma": "Caatinga",
        "risco": "Moderado"
    },
    {
        "nome": "🐍 Cascavel",
        "lat": -8.0,
        "lon": -39.0,
        "bioma": "Caatinga",
        "risco": "Baixo"
    },
    {
        "nome": "🦜 Ararinha-azul",
        "lat": -9.0,
        "lon": -42.5,
        "bioma": "Caatinga",
        "risco": "Crítico"
    },
    {
        "nome": "🐍 Jararaca",
        "lat": -7.5,
        "lon": -37.5,
        "bioma": "Caatinga",
        "risco": "Alto"
    },
    {
        "nome": "🦎 Calango",
        "lat": -8.8,
        "lon": -41.0,
        "bioma": "Caatinga",
        "risco": "Baixo"
    },
    {
        "nome": "🐗 Porco-do-mato",
        "lat": -10.5,
        "lon": -38.5,
        "bioma": "Caatinga",
        "risco": "Moderado"
    },
    {
        "nome": "🐦 Asa-branca",
        "lat": -6.2,
        "lon": -36.0,
        "bioma": "Caatinga",
        "risco": "Alto"
    },

    # --- PAMPA ---
    {
        "nome": "🦌 Veado-campeiro",
        "lat": -31.5,
        "lon": -53.5,
        "bioma": "Pampa",
        "risco": "Alto"
    },
    {
        "nome": "🐈 Gato-palheiro",
        "lat": -30.0,
        "lon": -54.0,
        "bioma": "Pampa",
        "risco": "Moderado"
    },
    {
        "nome": "🦆 Pato-mergulhão",
        "lat": -30.5,
        "lon": -52.5,
        "bioma": "Pampa",
        "risco": "Crítico"
    },
    {
        "nome": "🐾 Gato-dos-pampas",
        "lat": -32.0,
        "lon": -55.5,
        "bioma": "Pampa",
        "risco": "Alto"
    },
    {
        "nome": "🦌 Veado-virá",
        "lat": -29.5,
        "lon": -56.0,
        "bioma": "Pampa",
        "risco": "Moderado"
    },
    {
        "nome": "🐍 Boipeva",
        "lat": -31.8,
        "lon": -53.2,
        "bioma": "Pampa",
        "risco": "Baixo"
    }
]

    # ========================================================
    # 📍 CLUSTER DE MARCADORES
    # ========================================================

    marker_cluster = MarkerCluster().add_to(mapa)

    # ========================================================
    # 🎨 CORES DE RISCO
    # ========================================================

    cores_risco = {
        "Baixo": "green",
        "Moderado": "orange",
        "Alto": "red",
        "Crítico": "darkred"
    }

    # ========================================================
    # 🐾 ADICIONAR ESPÉCIES
    # ========================================================

    for especie in especies:

        popup_html = f"""
        <div style="width:200px">

        <h4>{especie['nome']}</h4>

        <b>🌱 Bioma:</b> {especie['bioma']}<br>

        <b>⚠️ Risco:</b> {especie['risco']}<br>

        </div>
        """

        folium.CircleMarker(
            location=[especie["lat"], especie["lon"]],
            radius=12,
            popup=popup_html,
            color=cores_risco[especie["risco"]],
            fill=True,
            fill_color=cores_risco[especie["risco"]],
            fill_opacity=0.9
        ).add_to(marker_cluster)

    # ========================================================
    # 🌱 LEGENDA
    # ========================================================

    legenda = """
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        width: 230px;
        height: 200px;
        background-color: white;
        border:2px solid grey;
        z-index:9999;
        font-size:14px;
        padding: 10px;
        border-radius:10px;
    ">

    <h4>🌎 BioGuard</h4>

    <b>🔥 Heatmap</b><br>
    Intensidade de queimadas<br><br>

    <b>🐾 Espécies</b><br>

    🟢 Baixo<br>
    🟠 Moderado<br>
    🔴 Alto<br>
    ⚫ Crítico

    </div>
    """

    mapa.get_root().html.add_child(
        folium.Element(legenda)
    )

    # ========================================================
    # 💾 EXPORTAR
    # ========================================================

    mapa.save("mapa.html")

    print("✅ Mapa salvo!")


Latitude: lat
Longitude: lon
🔥 Registros usados: 277981
✅ Mapa salvo!



# 📊 Dashboard — Queimadas por bioma


In [19]:
if coluna_bioma:

    fig = px.histogram(
        df,
        x=coluna_bioma,
        color=coluna_bioma,
        title="🔥 Queimadas por Bioma"
    )

    fig.show()



# 🐾 Dashboard — Espécies monitoradas


In [20]:
fig = px.pie(
    df_especies,
    names="categoria_iucn",
    title="🐾 Distribuição das Categorias IUCN"
)

fig.show()



# 📄 Relatório ambiental automático


In [21]:
display(Markdown(f'''
# 🌎 Relatório Ambiental — BioGuard

## 🔥 Queimadas monitoradas
**{qtd_queimadas}** focos detectados.

## ⚠️ Nível de risco ambiental
**{nivel}**

## 🐾 Espécies monitoradas
{", ".join(df_especies["nome_comum"].tolist())}

## 📊 Categorias IUCN
{df_especies[["nome_comum", "categoria_iucn"]].to_markdown(index=False)}

---

## 🌱 Conclusão

O BioGuard utiliza dados espaciais do INPE para identificar impactos ambientais na fauna brasileira, conectando queimadas e espécies ameaçadas da Amazônia e Pantanal.

O sistema permite visualizar regiões críticas, analisar riscos ambientais e auxiliar estratégias de preservação da biodiversidade.
'''))



# 🌎 Relatório Ambiental — BioGuard

## 🔥 Queimadas monitoradas
**277981** focos detectados.

## ⚠️ Nível de risco ambiental
**Crítico**

## 🐾 Espécies monitoradas
Onça-pintada, Arara-azul, Tamanduá-bandeira, Lobo-guará, Jacaré-do-pantanal, Tuiuiú, Capivara, Anta, Ariranha, Jaguatirica, Cutia, Queixada, Ema, Cágado-de-barbiga, Jacaré-açu, Sauim-de-coleira, Macaco-aranha, Uacari-branco, Boto-cor-de-rosa, Ariranha, Gavião-real, Tartaruga-da-amazônia, Guariba-vermelho, Macaco-prego, Irara, Morcego-lanceolado, Pirarucu, Rã-venenosa, Tamanduá-bandeira, Onça-pintada, Lobo-guará, Raposa-do-campo, Veado-campeiro, Gato-do-mato-pequeno, Ema, Gralha-do-cerrado, Jacaré-do-papo-amarelo, Anta, Capivara, Queixada, Cutia, Cascavel, Irara, Bugio-preto, Onça-parda, Mico-leão-dourado, Muriqui-do-sul, Araponga, Gato-do-mato, Tucanuçu, Gambá-de-orelha-preta, Sagui, Tamanduá-mirim, Peixe-boi-marinho, Gavião-pombo, Pitiguari, Onça-parda, Cutia, Perereca-gigante, Bugio-ruivo, Sagui-do-nordeste, Raposa-do-campo, Gato-do-mato, Perereca, Suiriri, Cascavel, Anta, Capivara, Jacaré, Cutia, Ema, Morcego-de-focinho-curto, Queixada, Veado-campeiro, Piranha-vermelha, Veado-campeiro, Ema-do-pampa, Lebre-da-patagônia, Puma, Jaguatirica, Capivara, Cutia, Lobo-guará, Anta, Jacaré, Cágado, Morcego, Cascavel, Ema, Queixada

## 📊 Categorias IUCN
| nome_comum               | categoria_iucn   |
|:-------------------------|:-----------------|
| Onça-pintada             | NT               |
| Arara-azul               | VU               |
| Tamanduá-bandeira        | VU               |
| Lobo-guará               | NT               |
| Jacaré-do-pantanal       | LC               |
| Tuiuiú                   | LC               |
| Capivara                 | LC               |
| Anta                     | VU               |
| Ariranha                 | EN               |
| Jaguatirica              | LC               |
| Cutia                    | LC               |
| Queixada                 | NT               |
| Ema                      | LC               |
| Cágado-de-barbiga        | LC               |
| Jacaré-açu               | LC               |
| Sauim-de-coleira         | CR               |
| Macaco-aranha            | VU               |
| Uacari-branco            | VU               |
| Boto-cor-de-rosa         | EN               |
| Ariranha                 | EN               |
| Gavião-real              | NT               |
| Tartaruga-da-amazônia    | VU               |
| Guariba-vermelho         | LC               |
| Macaco-prego             | LC               |
| Irara                    | LC               |
| Morcego-lanceolado       | LC               |
| Pirarucu                 | EN               |
| Rã-venenosa              | LC               |
| Tamanduá-bandeira        | VU               |
| Onça-pintada             | NT               |
| Lobo-guará               | NT               |
| Raposa-do-campo          | NT               |
| Veado-campeiro           | VU               |
| Gato-do-mato-pequeno     | VU               |
| Ema                      | LC               |
| Gralha-do-cerrado        | LC               |
| Jacaré-do-papo-amarelo   | LC               |
| Anta                     | VU               |
| Capivara                 | LC               |
| Queixada                 | NT               |
| Cutia                    | LC               |
| Cascavel                 | LC               |
| Irara                    | LC               |
| Bugio-preto              | NT               |
| Onça-parda               | LC               |
| Mico-leão-dourado        | EN               |
| Muriqui-do-sul           | CR               |
| Araponga                 | VU               |
| Gato-do-mato             | VU               |
| Tucanuçu                 | LC               |
| Gambá-de-orelha-preta    | LC               |
| Sagui                    | LC               |
| Tamanduá-mirim           | LC               |
| Peixe-boi-marinho        | VU               |
| Gavião-pombo             | LC               |
| Pitiguari                | LC               |
| Onça-parda               | LC               |
| Cutia                    | LC               |
| Perereca-gigante         | LC               |
| Bugio-ruivo              | CR               |
| Sagui-do-nordeste        | LC               |
| Raposa-do-campo          | NT               |
| Gato-do-mato             | VU               |
| Perereca                 | LC               |
| Suiriri                  | LC               |
| Cascavel                 | LC               |
| Anta                     | VU               |
| Capivara                 | LC               |
| Jacaré                   | LC               |
| Cutia                    | LC               |
| Ema                      | LC               |
| Morcego-de-focinho-curto | LC               |
| Queixada                 | NT               |
| Veado-campeiro           | VU               |
| Piranha-vermelha         | LC               |
| Veado-campeiro           | VU               |
| Ema-do-pampa             | NT               |
| Lebre-da-patagônia       | LC               |
| Puma                     | LC               |
| Jaguatirica              | LC               |
| Capivara                 | LC               |
| Cutia                    | LC               |
| Lobo-guará               | NT               |
| Anta                     | VU               |
| Jacaré                   | LC               |
| Cágado                   | LC               |
| Morcego                  | LC               |
| Cascavel                 | LC               |
| Ema                      | LC               |
| Queixada                 | NT               |

---

## 🌱 Conclusão

O BioGuard utiliza dados espaciais do INPE para identificar impactos ambientais na fauna brasileira, conectando queimadas e espécies ameaçadas da Amazônia e Pantanal.

O sistema permite visualizar regiões críticas, analisar riscos ambientais e auxiliar estratégias de preservação da biodiversidade.



# 💾 Exportação dos resultados


In [22]:

# ============================================================
# 💾 EXPORTAR RESULTADOS
# ============================================================

df.to_csv("dados_filtrados_bioguard.csv", index=False)

df_especies.to_csv("especies_monitoradas.csv", index=False)

print("✅ Arquivos exportados com sucesso!")


✅ Arquivos exportados com sucesso!
